# Notebook 26 — Q4 feature-only scoring and score lock

Run after Notebook 25 completed successfully. Install the same `torch`, `scikit-learn`, `joblib` and Parquet support used in earlier notebooks. Adjust only `COMPONENT_ROOT` if needed. The frozen candidate and thresholds cannot be changed. This notebook reads only five SMART fields, drive ID, model and date; it never requests the `failure` column. A complete 30-day valid window is required for every score and all eight methods use identical endpoints. Daily partitions are resumable and checked against the input hashes and scored keys. Keep the resulting `q4_score_lock.json` intact before opening Notebook 27. Large Q4 archives may take time.

In [2]:
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib, json, re, zipfile
import numpy as np
import pandas as pd
import joblib
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
LOCK=COMPONENT_ROOT/'results'/RUN_ID/'q4_blind_candidate_protocol_v1'
OUT=COMPONENT_ROOT/'results'/RUN_ID/'q4_feature_only_scoring_v1'
PARTS=OUT/'daily_scores'
PARTS.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
Q4_ZIP=COMPONENT_ROOT/'data/raw/data_Q4_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
def freeze_json(path,obj):
    if path.exists():assert read_json(path)==obj,f'Frozen artifact differs: {path}'
    else:path.write_text(json.dumps(obj,indent=2))
lock_path=LOCK/'q4_candidate_lock.json'
complete=read_json(LOCK/'q4_protocol_lock_complete.json')
lock=read_json(lock_path);protocol=lock['protocol']
assert complete['analysis_complete'] and sha256(lock_path)==complete['candidate_lock_sha256']
assert sha256(LOCK/'q4_blind_protocol.json')==complete['protocol_sha256']
assert not lock['q4_outcomes_opened'] and not lock['q4_archive_opened']
for key,item in lock['artifacts'].items():
    assert sha256(COMPONENT_ROOT/item['path'])==item['sha256'],key
assert protocol['primary_method']=='transformer_sector_error'
assert protocol['window_days']==30 and protocol['score_lock_before_labels']
assert protocol['scoring_endpoint_dates']==['2021-10-01','2021-12-31']
assert sha256(Q3_ZIP)==protocol['q3_archive_sha256']
assert Q4_ZIP.is_file()
models=list(lock['future_test_drives_by_model']);assert len(models)==5
reservation=read_json(COMPONENT_ROOT/lock['artifacts']['reservation']['path'])
future={str(s):m for m,g in reservation['models'].items() for s in g['future_test']}
assert {m:sum(x==m for x in future.values()) for m in models}==lock['future_test_drives_by_model']
FEATURES=['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
F_NAMES=['last_'+f for f in FEATURES]+['std30_'+f for f in FEATURES]+['positive7_'+f for f in ['smart_5_raw','smart_197_raw','smart_198_raw']]
WARM=pd.date_range('2021-09-02','2021-09-30').strftime('%Y-%m-%d').tolist()
DATES=pd.date_range('2021-10-01','2021-12-31').strftime('%Y-%m-%d').tolist()
ALL=WARM+DATES
DAY={date:i for i,date in enumerate(ALL)}
q4_digest=sha256(Q4_ZIP)
settings={'candidate_lock_sha256':sha256(lock_path),'q3_archive_sha256':sha256(Q3_ZIP),
          'q4_archive_sha256':q4_digest,'warmup_start':WARM[0],
          'q4_dates':[DATES[0],DATES[-1]],'features':FEATURES,
          'window_days':30,'failure_column_read':False,'batch_size':256}
freeze_json(OUT/'settings.json',settings)
print('Candidate artifacts verified. Q4 feature archive SHA:',q4_digest)
print('Failure labels have not been loaded.')

Candidate artifacts verified. Q4 feature archive SHA: e8753f153054823237f50a7abbae0d39f057122b12aec60f2b34278c602e749e
Failure labels have not been loaded.


## Frozen models and exact Q3 scorer

In [3]:
class MaskedTelemetryTransformer(nn.Module):
    def __init__(self,n_models,width=64,heads=4,layers=2,feedforward=128,dropout=.1):
        super().__init__()
        self.input=nn.Linear(15,width)
        self.model_embedding=nn.Embedding(n_models,width)
        self.position=nn.Parameter(torch.zeros(1,30,width))
        nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=feedforward,
                                          dropout=dropout,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(block,num_layers=layers,enable_nested_tensor=False)
        self.output=nn.Linear(width,5)
    def forward(self,x,model_id,hide):
        observed=(~hide).to(x.dtype)
        masked=x.masked_fill(hide,0)
        delta=torch.zeros_like(x)
        delta[:,1:,:]=x[:,1:,:]-x[:,:-1,:]
        delta_visible=observed.clone()
        delta_visible[:,1:,:]=observed[:,1:,:]*observed[:,:-1,:]
        delta=delta*delta_visible
        h=self.input(torch.cat([masked,observed,delta],dim=-1))
        h=h+self.model_embedding(model_id)[:,None,:]+self.position
        return self.output(self.encoder(h))


train_settings=read_json(COMPONENT_ROOT/lock['artifacts']['transformer_training_settings']['path'])
prep_path=COMPONENT_ROOT/lock['artifacts']['transformer_preprocessing']['path']
checkpoint_path=COMPONENT_ROOT/lock['artifacts']['transformer_checkpoint']['path']
prep=read_json(prep_path)
assert prep['features']==FEATURES and prep['models']==train_settings['models']==models
model_index={m:i for i,m in enumerate(models)}
HIDDEN=[0,2,3,4]
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
network=MaskedTelemetryTransformer(len(models),train_settings['width'],train_settings['heads'],
    train_settings['layers'],train_settings['feedforward'],train_settings['dropout']).to(DEVICE)
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
assert checkpoint['models']==models and checkpoint['preprocessing_sha256']==sha256(prep_path)
network.load_state_dict(checkpoint['state_dict']);network.eval()
fitted=joblib.load(COMPONENT_ROOT/lock['artifacts']['if_model']['path'])
assert set(fitted)==set(models)
for m in models:
    assert fitted[m]['threshold']==lock['thresholds']['isolation_forest'][m]
# The same four final-day values are hidden in both Q3 and Q4.
x=torch.arange(2*30*5,dtype=torch.float32,device=DEVICE).reshape(2,30,5)/100
ids=torch.tensor([0,1],device=DEVICE)
hide=torch.zeros_like(x,dtype=torch.bool);hide[:,-1,HIDDEN]=True
with torch.no_grad():
    original=network(x,ids,hide)
    changed=x.clone();changed[hide]+=3
    torch.testing.assert_close(original,network(changed,ids,hide),rtol=1e-6,atol=1e-6)
print('Frozen transformer, preprocessing, IF bundle and leakage check verified.')
def normalized(windows,model_names):
    result=np.empty_like(windows,dtype=np.float32)
    logged=np.log1p(windows.astype(np.float64))
    for model in models:
        chosen=np.asarray(model_names)==model
        if chosen.any():
            center=np.asarray(prep['center'][model],dtype=np.float64)
            scale=np.asarray(prep['scale'][model],dtype=np.float64)
            result[chosen]=((logged[chosen]-center)/scale).astype(np.float32)
    assert np.isfinite(result).all()
    return result

@torch.no_grad()
def predict_errors(windows,model_names):
    if len(windows)==0:return np.empty((0,4),dtype=np.float32)
    scaled=normalized(windows,model_names)
    ids=np.array([model_index[x] for x in model_names],dtype=np.int64)
    loader=DataLoader(TensorDataset(torch.from_numpy(scaled),torch.from_numpy(ids)),
                      batch_size=256,shuffle=False,num_workers=0)
    collected=[]
    for x,m in loader:
        x=x.to(DEVICE);m=m.to(DEVICE)
        hide=torch.zeros_like(x,dtype=torch.bool);hide[:,-1,HIDDEN]=True
        residual=(network(x,m,hide)[:,-1,HIDDEN]-x[:,-1,HIDDEN]).square()
        collected.append(residual.cpu().numpy())
    result=np.concatenate(collected).astype(np.float32)
    assert result.shape==(len(windows),4) and np.isfinite(result).all()
    return result



Frozen transformer, preprocessing, IF bundle and leakage check verified.


## Stream September warmup and October–December feature rows

In [4]:
def members_for(z,dates):
    result={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in dates:
            assert match.group(1) not in result,'Duplicate ZIP date'
            result[match.group(1)]=info.filename
    assert set(result)==set(dates),f'Missing daily files: {set(dates)-set(result)}'
    return result

def featurize(window):
    a=np.asarray(window,dtype=np.float64)
    assert a.shape==(30,5)
    return np.concatenate([np.log1p(a[-1]),np.log1p(a.std(axis=0)),
        np.log1p(np.maximum(a[-1,[0,3,4]]-a[-8,[0,3,4]],0))]).astype(np.float32)

history={};last={};counts={};hashes={};total=0
with zipfile.ZipFile(Q3_ZIP) as q3,zipfile.ZipFile(Q4_ZIP) as q4:
    m3=members_for(q3,WARM);m4=members_for(q4,DATES)
    for date in ALL:
        day=DAY[date];z=q3 if date in m3 else q4
        member=(m3 if date in m3 else m4)[date]
        seen=set();records=[];observed=0;invalid=0;gaps=0
        with z.open(member) as f:
            # Explicit usecols prevents loading failure, including Q3 warmup.
            for chunk in pd.read_csv(f,usecols=['date','serial_number','model']+FEATURES,
                                     dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(future)]
                if part.empty:continue
                assert part.date.astype(str).eq(date).all(),'Filename/date mismatch'
                serials=part.serial_number.tolist()
                assert len(serials)==len(set(serials)) and seen.isdisjoint(serials),'Duplicate drive/day'
                seen.update(serials);observed+=len(serials)
                assert all(future[s]==m for s,m in part[['serial_number','model']].itertuples(index=False,name=None))
                values=part[FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64)
                for serial,v in zip(serials,values):
                    if not(np.isfinite(v).all() and (v>=0).all()):
                        invalid+=1;history.pop(serial,None);last[serial]=day;continue
                    if last.get(serial,-2)!=day-1:
                        history[serial]=deque(maxlen=30)
                        if serial in last:gaps+=1
                    h=history.setdefault(serial,deque(maxlen=30))
                    h.append(v.astype(np.float32));last[serial]=day
                    if len(h)==30 and date in m4:
                        window=np.stack(h)
                        sector=window[-1,[0,3,4]]
                        delta=float(np.maximum(sector-window[-2,[0,3,4]],0).sum())
                        records.append((serial,future[serial],window,float(np.max(sector)),delta))
        if date in m3:continue
        records.sort(key=lambda x:x[0]);keys=[(s,m,date) for s,m,*_ in records]
        expected_rows=len(keys)
        identity=hashlib.sha256((sha256(OUT/'settings.json')+date).encode()).hexdigest()
        path=PARTS/f'{date}.parquet';meta=PARTS/f'{date}.json'
        stats={'observed_rows':observed,'invalid_rows':invalid,'gap_resets':gaps,'scored_rows':expected_rows,
               'scored_by_model':{m:sum(k[1]==m for k in keys) for m in models}}
        assert path.exists()==meta.exists(),f'Incomplete cached partition: {date}'
        if path.exists():
            saved=read_json(meta)
            assert saved['identity']==identity and saved['stats']==stats
            assert saved['sha256']==sha256(path)
            check=pd.read_parquet(path,columns=['serial_number','model','end_date'])
            assert list(check.itertuples(index=False,name=None))==keys
        else:
            windows=np.stack([r[2] for r in records]) if records else np.empty((0,30,5),dtype=np.float32)
            errors=predict_errors(windows,np.asarray([r[1] for r in records]))
            table=pd.DataFrame(keys,columns=['serial_number','model','end_date'])
            table['sector_error']=errors[:,[0,2,3]].mean(axis=1) if len(errors) else np.array([],dtype=np.float32)
            table['temp_error']=errors[:,1]
            table['joint_error']=errors.mean(axis=1) if len(errors) else np.array([],dtype=np.float32)
            table['sector_positive']=[r[3] for r in records]
            table['sector_delta']=[r[4] for r in records]
            table['isolation_forest']=np.nan
            for m in models:
                positions=np.flatnonzero(table.model.to_numpy()==m)
                if len(positions):
                    X=np.stack([featurize(records[i][2]) for i in positions])
                    bundle=fitted[m]
                    table.loc[positions,'isolation_forest']=-bundle['detector'].score_samples(bundle['scaler'].transform(X))
            assert np.isfinite(table[['sector_error','temp_error','joint_error','sector_positive','sector_delta','isolation_forest']].to_numpy(dtype=float)).all()
            threshold=lock['thresholds']
            for col in ['sector_error','temp_error','joint_error']:
                table['transformer_'+col]=table[col]>table.model.map({m:threshold['transformer'][m][col] for m in models})
            table['isolation_forest_flag']=table.isolation_forest>table.model.map(threshold['isolation_forest'])
            for col in ['sector_positive','sector_delta']:
                table[col+'_fixed']=table[col]>0
                table[col+'_calibrated']=table[col]>table.model.map({m:threshold['smart'][m][col] for m in models})
            table.to_parquet(path,index=False)
            meta.write_text(json.dumps({'identity':identity,'sha256':sha256(path),'stats':stats},indent=2))
        counts[date]=stats;hashes[path.name]=sha256(path);total+=expected_rows
        if date.endswith(('-01','-15')):print('Q4 feature scored:',date,expected_rows)
assert len(hashes)==len(DATES)==92
assert sum(x['scored_rows'] for x in counts.values())==total
print('Feature-only Q4 score rows:',total)

Q4 feature scored: 2021-10-01 19155
Q4 feature scored: 2021-10-15 19146
Q4 feature scored: 2021-11-01 19159
Q4 feature scored: 2021-11-15 19072
Q4 feature scored: 2021-12-01 19042
Q4 feature scored: 2021-12-15 19136
Feature-only Q4 score rows: 1758728


## Freeze scores before reading outcomes

In [5]:
manifest={'settings_sha256':sha256(OUT/'settings.json'),'candidate_lock_sha256':sha256(lock_path),
          'q4_archive_sha256':q4_digest,'partition_sha256':hashes,'daily_quality':counts,
          'score_rows':total,'failure_column_read':False,'q4_failure_outcomes_opened':False}
manifest_path=OUT/'q4_score_manifest.json';freeze_json(manifest_path,manifest)
score_lock={'candidate_lock_sha256':sha256(lock_path),'score_manifest_sha256':sha256(manifest_path),
            'q4_archive_sha256':q4_digest,'score_rows':total,
            'failure_column_read':False,'q4_failure_outcomes_opened':False}
score_lock_path=OUT/'q4_score_lock.json';freeze_json(score_lock_path,score_lock)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'score_lock_sha256':sha256(score_lock_path),'score_manifest_sha256':sha256(manifest_path),
        'candidate_lock_sha256':sha256(lock_path),'q4_archive_sha256':q4_digest,
        'score_rows':total,'q4_failure_outcomes_opened':False,'sealed_June_test_opened':False,
        'next_gate':'Verify score lock, then read Q4 failure labels once in Notebook 27'}
completion_path=OUT/'q4_scoring_complete.json'
if completion_path.exists():
    earlier=read_json(completion_path)
    assert {k:v for k,v in earlier.items() if k!='created_utc'}=={k:v for k,v in record.items() if k!='created_utc'}
else:completion_path.write_text(json.dumps(record,indent=2))
print('NOTEBOOK 26 COMPLETE — scores and score lock frozen before failure labels.')

NOTEBOOK 26 COMPLETE — scores and score lock frozen before failure labels.
